**Building BI ready tables**

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType
from pyspark.sql import Row

catalog_name="ecommerce"

adding more dimension to data- category code--- CATEGORY NAME// Brand code--- BRAND NAME

In [0]:
df_products=spark.read.table(f"{catalog_name}.silver.slv_products")
df_brands=spark.read.table(f"{catalog_name}.silver.slv_brands")
df_category=spark.read.table(f"{catalog_name}.silver.slv_category")

for Joins- views are better- dataframes will be more like tables

In [0]:
df_products.createOrReplaceTempView("v_products")
df_brands.createOrReplaceTempView("v_brands")
df_category.createOrReplaceTempView("v_category")

In [0]:
display(spark.sql("select * from v_products limit 5"))

In [0]:
display(spark.sql("select * from v_brands limit 5"))

In [0]:
display(spark.sql("select * from v_category limit 5"))

In [0]:
#making sure we are on right catalog
spark.sql(f"use catalog {catalog_name}")

In [0]:
%sql

--build brand*category mapping and write gold table
--CTE QUERY

CREATE OR REPLACE TABLE gold.gld_dim_products AS

WITH brands_categories AS (
  SELECT
  b.brand_name,
  b.brand_code,
  c.category_name,
  c.category_code
  FROM v_brands b
  INNER JOIN v_category c
  ON 
  b.category_code=c.category_code
)

SELECT
p.product_id,
p.sku,
p.category_code,
COALESCE(bc.category_name, "Not Available") AS category_name,
p.brand_code,
COALESCE(bc.brand_name, "Not Available") AS brand_name,
p.color,
p.size,
p.material,
p.weight_grams,
p.length_cm,
p.width_cm,
p.height_cm,
p.rating_count,
p._source_file,
p._ingested_at
FROM v_products p
LEFT JOIN brands_categories bc
ON p.brand_code= bc.brand_code

Customers table

In [0]:
#India states
India_region={
    "MH":"West","GJ":"West","RJ":"West",
    "UP":"North","WB":"North","DL":"North",
    "KL":"South","TN":"South","KA":"South","AP":"South","TS":"South"

}

#Australia states
Australia_region={
    "VIC":"SouthEast","WA":"West","NSW":"East","QLD":"NorthEast"

}

#USA states
US_region={
    "MA":"NorthEast","FL":"South","NY":"NorthEast","TX":"South",
    "NJ":"NorthEast","CA":"West"
}

#UK states
UK_region={
    "ENG":"England","WLS":"Wales","NIR":"Northern Ireland","SCT":"Scotland"
}

#uae states
UAE_region={
    "AUH":"Abu Dhabi","DU":"Dubai","SHJ":"Sharjah"
}
#singapore states
Singapore_region={
    "SG":"Singapore"
}
#canada states
Canada_region={
    "BC":"West","AB":"West","ON":"East","QC":"East","NS":"East","IL":"Other"
}

#combine to a master dict
country_state_map={
    "India":India_region,
    "Australia":Australia_region,
    "USA":US_region,
    "UK":UK_region,
    "UAE":UAE_region,
    "Singapore":Singapore_region,
    "Canada":Canada_region
}

In [0]:
country_state_map

flatten country_state_map into a list of rows

In [0]:
rows=[]
for country,states in country_state_map.items():
    for state_code,region in states.items():
        rows.append(Row(country=country,state=state_code,region=region))
rows[:10]

In [0]:
#create mapping dataframe
df_region_mapping=spark.createDataFrame(rows)

#show mapping
df_region_mapping.show(truncate=False)

In [0]:
df_silver_cust=spark.read.table(f"{catalog_name}.silver.slv_customers")
df_silver_cust.show(4)

In [0]:
df_gold_cust=df_silver_cust.join(df_region_mapping,on=["country","state"],how="left")

df_gold_cust=df_gold_cust.fillna({'region':"Other"})

display(df_gold_cust.limit(5))

In [0]:
df_gold_cust.write.format("delta").mode("overwrite").option("mergeSchema","true").\
    saveAsTable(f"{catalog_name}.gold.gld_customers")

**Calender/Date**

In [0]:
df_slvr_dte=spark.read.table(f"{catalog_name}.silver.slv_date")
display(df_slvr_dte.limit(5))

In [0]:
df_gld_dte=df_slvr_dte.withColumn("date_id",F.date_format(F.col("date"), "yyyyMMdd").cast("int"))

#add month name January, etc

df_gld_dte=df_gld_dte.withColumn("month_name",F.date_format(F.col("date"), "MMMM"))

#add is weekend column

df_gld_dte=df_gld_dte.withColumn(
    "is_weekend",
    F.when(F.col("day_name").isin("Saturday","Sunday"),1).otherwise(0)
)

display(df_gld_dte.limit(5))

In [0]:
desired_columns_order=["date_id","date","year","month_name","day_name","is_weekend","quarter","week","_source_file","_ingested_at"]
df_gld_dte=df_gld_dte.select(desired_columns_order)

display(df_gld_dte.limit(5))

In [0]:
df_gld_dte.write.format("delta").mode("overwrite").option("mergeSchema","true").\
    saveAsTable(f"{catalog_name}.gold.gld_date")